In [1]:
import sys

print(sys.executable)

/Users/quaxiom/Documents/DEVLOPMENT/ML_CORE/AI_Venv/bin/python


In [2]:
import os

from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/_j/6cf49_l57txgpk2fdvfkj5n80000gn/T/ipykernel_44226/1024066971.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/Users/quaxiom/Documents/DEVLOPMENT/ML_CORE/AI_Venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 13 PDF files to process

Processing: M1.pdf
  ✓ Loaded 49 pages

Processing: M3.pdf
  ✓ Loaded 58 pages

Processing: Cormen Introduction to Algorithms.pdf
  ✓ Loaded 1313 pages

Processing: Module 2_Data Preprocessing.pdf
  ✓ Loaded 63 pages

Processing: data-mining-concepts-and-techniques-3rd-edition_compress.pdf
  ✓ Loaded 560 pages

Processing: QuAxiom_Personal_Introduction.pdf
  ✓ Loaded 1 pages

Processing: UniAssist_Self_Introduction.pdf
  ✓ Loaded 1 pages

Processing: ML Machine Learning-A Probabilistic Perspective.pdf
  ✓ Loaded 1098 pages

Processing: M2.1.pdf
  ✓ Loaded 103 pages

Processing: Module 1_Session 1.pdf
  ✓ Loaded 21 pages

Processing: Module 1_Session 2.pdf
  ✓ Loaded 11 pages

Processing: Computer-Networks-Global-Edition-by-Andrew-Tanenbaum-Nick-Feamster-David-Wetherall.pdf
  ✓ Loaded 946 pages

Processing: Module 1_Session 3.pdf
  ✓ Loaded 10 pages

Total documents loaded: 4234


In [4]:
all_pdf_documents


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'source_file': 'M1.pdf', 'file_type': 'pdf'}, page_content='Amity Institute of Information Technology\nINTRODUCTION TO DATA COMMUNICATION AND COMPUTER NETWORK (IDCCN)\n[CSIT369]\nMODULE 1  \nINTRODUCTION TO SIGNALS\nDr. Nidhi\nDr. Sudhanshu\nDr. Gurpreet\nDr. Sandhya\nDr. Vivek\n1\n(UG - III Sem)'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 1, 'page_label': '2', 'source_file': 'M1.pdf', 'file_ty

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 4234 documents into 12601 chunks

Example chunk:
Content: Amity Institute of Information Technology
INTRODUCTION TO DATA COMMUNICATION AND COMPUTER NETWORK (IDCCN)
[CSIT369]
MODULE 1  
INTRODUCTION TO SIGNALS
Dr. Nidhi
Dr. Sudhanshu
Dr. Gurpreet
Dr. Sandhya
...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'source_file': 'M1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'source_file': 'M1.pdf', 'file_type': 'pdf'}, page_content='Amity Institute of Information Technology\nINTRODUCTION TO DATA COMMUNICATION AND COMPUTER NETWORK (IDCCN)\n[CSIT369]\nMODULE 1  \nINTRODUCTION TO SIGNALS\nDr. Nidhi\nDr. Sudhanshu\nDr. Gurpreet\nDr. Sandhya\nDr. Vivek\n1\n(UG - III Sem)'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 1, 'page_label': '2', 'source_file': 'M1.pdf', 'file_ty

In [7]:
import sys

!{sys.executable} -m pip install -U sentence-transformers

In [8]:
from sentence_transformers import SentenceTransformer

print("sentence-transformers loaded successfully")

sentence-transformers loaded successfully


In [9]:
import sys

print(sys.executable)

import chromadb
import sentence_transformers

print("Chroma:", chromadb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

/Users/quaxiom/Documents/DEVLOPMENT/ML_CORE/AI_Venv/bin/python
Chroma: 1.5.9
Sentence Transformers: 6.0.0


In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
import sqlite3

print("SQLite:", sqlite3.sqlite_version)

conn = sqlite3.connect(":memory:")

for option in conn.execute("PRAGMA compile_options;").fetchall():
    if "MAX_VARIABLE_NUMBER" in option[0]:
        print(option[0])

SQLite: 3.51.2
MAX_VARIABLE_NUMBER=250000


In [12]:
import inspect
import chromadb.api.rust

print(inspect.getsource(chromadb.api.rust.RustBindingsAPI.get_max_batch_size))


    @override
    def get_max_batch_size(self) -> int:
        return self.bindings.get_max_batch_size()



In [13]:
import chromadb.api.rust

print(chromadb.api.rust.__file__)

/Users/quaxiom/Documents/DEVLOPMENT/ML_CORE/AI_Venv/lib/python3.13/site-packages/chromadb/api/rust.py


In [14]:
import chromadb_rust_bindings
import inspect

print(chromadb_rust_bindings.__file__)
print(dir(chromadb_rust_bindings.Bindings))


/Users/quaxiom/Documents/DEVLOPMENT/ML_CORE/AI_Venv/lib/python3.13/site-packages/chromadb_rust_bindings/__init__.py
['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'add', 'count', 'count_collections', 'create_collection', 'create_database', 'create_tenant', 'delete', 'delete_collection', 'delete_database', 'get', 'get_collection', 'get_collection_by_id', 'get_database', 'get_max_batch_size', 'get_tenant', 'get_version', 'heartbeat', 'list_collections', 'list_databases', 'query', 'reset', 'update', 'update_collection', 'upsert']


In [15]:
import chromadb_rust_bindings

print(chromadb_rust_bindings.__file__)
print(dir(chromadb_rust_bindings))

/Users/quaxiom/Documents/DEVLOPMENT/ML_CORE/AI_Venv/lib/python3.13/site-packages/chromadb_rust_bindings/__init__.py
['Bindings', 'MigrationHash', 'MigrationMode', 'PythonBindingsConfig', 'SqliteDBConfig', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'chromadb_rust_bindings', 'cli']


In [16]:
import chromadb_rust_bindings

print(dir(chromadb_rust_bindings.Bindings))

['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'add', 'count', 'count_collections', 'create_collection', 'create_database', 'create_tenant', 'delete', 'delete_collection', 'delete_database', 'get', 'get_collection', 'get_collection_by_id', 'get_database', 'get_max_batch_size', 'get_tenant', 'get_version', 'heartbeat', 'list_collections', 'list_databases', 'query', 'reset', 'update', 'update_collection', 'upsert']


In [17]:
client = chromadb.PersistentClient(path="./chroma_db")

print(type(client._server))
print(type(client._server.bindings))

<class 'chromadb.api.rust.RustBindingsAPI'>
<class 'builtins.Bindings'>


In [18]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

print("Chroma:", chromadb.__version__)
print("Max batch size:", client.get_max_batch_size())

Chroma: 1.5.9
Max batch size: 5461


In [19]:
import chromadb_rust_bindings

print(chromadb_rust_bindings.Bindings.get_max_batch_size.__doc__)

None


In [20]:
import chromadb_rust_bindings

print(chromadb_rust_bindings.Bindings.__doc__)

In [21]:
import chromadb

print("Chroma version:", chromadb.__version__)

client = chromadb.PersistentClient(path="./chroma_db")

print("Client:", type(client))
print("Settings:", client.get_settings())

Chroma version: 1.5.9
Client: <class 'chromadb.api.client.Client'>
Settings: environment='' chroma_api_impl='chromadb.api.rust.RustBindingsAPI' chroma_server_nofile=None chroma_server_thread_pool_size=40 tenant_id='default' topic_namespace='default' chroma_server_host=None chroma_server_headers=None chroma_server_http_port=None chroma_server_ssl_enabled=False chroma_server_ssl_verify=None chroma_server_api_default_path=<APIVersion.V2: '/api/v2'> chroma_server_cors_allow_origins=[] chroma_http_keepalive_secs=40.0 chroma_http_max_connections=None chroma_http_max_keepalive_connections=None is_persistent=True persist_directory='./chroma_db' chroma_memory_limit_bytes=0 chroma_segment_cache_policy=None allow_reset=False chroma_auth_token_transport_header=None chroma_client_auth_provider=None chroma_client_auth_credentials=None chroma_server_auth_ignore_paths={'APIVersion.V2': ['GET'], 'APIVersion.V2/heartbeat': ['GET'], 'APIVersion.V2/version': ['GET'], 'APIVersion.V1': ['GET'], 'APIVersion.

In [22]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6883.14it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/_j/6cf49_l57txgpk2fdvfkj5n80000gn/T/ipykernel_44226/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [23]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Allow up to 12,000 documents while respecting Chroma's runtime limit.
        try:
            max_documents = 40000
            batch_size = min(max_documents, self.client.get_max_batch_size())
            for start in range(0, len(ids), batch_size):
                end = start + batch_size
                self.collection.add(
                    ids=ids[start:end],
                    embeddings=embeddings_list[start:end],
                    metadatas=metadatas[start:end],
                    documents=documents_text[start:end]
                )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 59978


In [24]:
chunks

[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'source_file': 'M1.pdf', 'file_type': 'pdf'}, page_content='Amity Institute of Information Technology\nINTRODUCTION TO DATA COMMUNICATION AND COMPUTER NETWORK (IDCCN)\n[CSIT369]\nMODULE 1  \nINTRODUCTION TO SIGNALS\nDr. Nidhi\nDr. Sudhanshu\nDr. Gurpreet\nDr. Sandhya\nDr. Vivek\n1\n(UG - III Sem)'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-07-28T11:21:44+05:30', 'title': '', 'author': 'Pooja Gambhir', 'moddate': '2026-07-28T11:21:44+05:30', 'source': '../data/M1.pdf', 'total_pages': 49, 'page': 1, 'page_label': '2', 'source_file': 'M1.pdf', 'file_ty

In [25]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 12601 texts...


Batches: 100%|██████████| 394/394 [00:34<00:00, 11.50it/s]


Generated embeddings with shape: (12601, 384)
Adding 12601 documents to vector store...
Successfully added 12601 documents to vector store
Total documents in collection: 72579


In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [27]:

rag_retriever

In [28]:
rag_retriever.retrieve("what is bigdata")

Retrieving documents for query: 'what is bigdata'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.22it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_953724c2_283',
  'content': 'AMITY INSTITUTE OF INFORMATION TECHNOLOGY\nWhat is Big Data?\nBig Data refers to datasets that are too large, fast, or complex to be\nprocessed using traditional database systems.\nIt is commonly characterized by the 5 Vs:\n• Volume – Huge amounts of data \n• Velocity – Rapid generation of data \n• Variety – Different types of data (structured, semi-structured, \nunstructured) \n• Veracity – Data quality and reliability \n• Value – Useful information obtained from the data \nExamples include social media posts, sensor data, online transactions, and \nIoT data. Technologies such as Hadoop, Spark, and NoSQL databases\nare commonly used to manage Big Data.',
  'metadata': {'total_pages': 21,
   'file_type': 'pdf',
   'page_label': '6',
   'source': '../data/Module 1_Session 1.pdf',
   'creator': 'Microsoft® PowerPoint® 2016',
   'content_length': 648,
   'doc_index': 283,
   'moddate': '2026-08-23T14:40:07+00:00',
   'page': 5,
   'creationdate': 

In [29]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [30]:
import sys

!{sys.executable} -m pip install -U langchain-groq


In [31]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [32]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [33]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [34]:
rag_retriever.retrieve("Amity institue of information tecnology?")

Retrieving documents for query: 'Amity institue of information tecnology?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d1e6a0f9_3702',
  'content': 'AMITY INSTITUTE OF INFORMATION TECHNOLOGY\nTHANK YOU',
  'metadata': {'total_pages': 10,
   'page_label': '10',
   'producer': 'www.ilovepdf.com',
   'creator': 'Microsoft® PowerPoint® 2016',
   'file_type': 'pdf',
   'doc_index': 3702,
   'page': 9,
   'source': '../data/Module 1_Session 3.pdf',
   'source_file': 'Module 1_Session 3.pdf',
   'creationdate': '2026-08-23T14:40:11+00:00',
   'moddate': '2026-08-23T14:40:11+00:00',
   'content_length': 51},
  'similarity_score': 0.13339293003082275,
  'distance': 0.8666070699691772,
  'rank': 1},
 {'id': 'doc_5cfed5ef_5716',
  'content': 'AMITY INSTITUTE OF INFORMATION TECHNOLOGY\nTHANK YOU',
  'metadata': {'doc_index': 5716,
   'creationdate': '2026-08-23T14:40:09+00:00',
   'page_label': '11',
   'total_pages': 11,
   'source': '../data/Module 1_Session 2.pdf',
   'content_length': 51,
   'producer': 'www.ilovepdf.com',
   'page': 10,
   'file_type': 'pdf',
   'source_file': 'Module 1_Session 2

In [35]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(

    api_key=groq_api_key,

    model_name="openai/gpt-oss-120b",

    temperature=0.1,

    max_tokens=1024

)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [36]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

canopylabs/orpheus-v1-english
groq/compound
whisper-large-v3
groq/compound-mini
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
whisper-large-v3-turbo
qwen/qwen3.6-27b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b


In [37]:
answer=rag_simple("What is Data Science?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Data Science?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 74.58it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Data Science is an interdisciplinary field that combines statistics, machine learning, programming, and domain expertise to analyze data—whether big or small—and solve real‑world problems.


In [38]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [39]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: The reply indicates that there is no pertinent information available. In other words, the necessary context is missing.
